# Sapient Notes

Collection of notes for Sapient development.

The plan is to produce a class or app that our sensors can plug into to generate Sapient compliant messages.

## Sapient

SAPIENT (Sensing for Asset Protection with Integrated Electronic Networked Technology) is a UK MOD-developed open standard for integrating sensors into autonomous sensing systems.

It defines a common communication protocol that allows different sensors, processing nodes and command-and-control systems to exchange information independently of manufacturer or hardware.

Key features:
- Interoperability: Sensors from different manufacturers can work together.
- Standardised messaging: **Uses Google Protocol Buffers** (Protobuf) to define structured messages.
- Network communication: Typically uses TCP/IP or UDP/IP.
- Autonomous operation: Sensors can report detections, tracks and status, and receive configuration or tasking commands.
In Python: A SAPIENT sensor node uses generated Protobuf classes to encode and decode messages, communicating with a SAPIENT-compliant system over a network.


## Building a Sapient Node

1) Start with Protobuf generation
2) Add TCP communinication


## Protobuf

Protocol Buffers (Protobuf) is a compact, efficient, binary data serialization format developed by Google. It uses .proto files to define structured messages, similar to C structs, with named fields, data types and unique field numbers. A Protobuf compiler generates code (e.g. Python or C++) to serialize these messages into binary data for transmission or storage, and deserialize them back into structured objects. Protobuf is language-independent, strongly typed and particularly useful for efficient communication between systems, such as SAPIENT sensor nodes.

## Python Enviroment

### Setup venv

```bash
python3.12 -m venv .venv3.12
source .venv3.12/bin/activate
```

### Install Dependencies

Following packages are required.

[protobuf](https://protobuf.dev/getting-started/pythontutorial/)  
[grpcio-tools](https://grpc.io/docs/languages/python/quickstart/)

To install dependancies:

```bash
pip install -e
```


### Protofiles
The offical Sapient protofiles were  cloned from DSTL and put in to the `sapient_msg` folder.

``` bash
git submodule add https://github.com/dstl/SAPIENT-Proto-Files.git sapient_msg
git submodule update --init --recursive
ls -la sapient_msg/bsi_flex_335_v2_0/
```

The correct folder structire

The protobuf files need to be compiled and can be done so using grpc_tools as follows:

```bash 
python -m grpc_tools.protoc -I=. --python_out=. sapient_msg/bsi_flex_335_v2_0/*.proto
python -m grpc_tools.protoc -I=. --python_out=. sapient_msg/*.proto
```

This generates lots of *_pb2.py files.


## How to Use the Files

### Sensor Registration

From the specification:

The message transfer sequence shall start with Initialization.
The Initialization process shall follow the sequence:  
1) A node connects as a client to the message handling application and sends a **registration message**;  
2) The message handling application sends a **registration acknowledgement message** to the node;  
3) The node sends an initial status message to the message handling application;  
NOTE At this point the message handling application or fusion node may optionally store relevant 
data from the message.  
4) A sensor edge node begins activity detection using its default tasking (for those sensor edge   
nodes which have defaults); and  
5)  An effector node awaits further tasking

----

Thus we need to build messages. 

Two broad message types:

1) Inner Messages:  Used by Outer Messages and have there own .protofiles. These include:  
 - Location
 - Range bearing
 - Velocity
 - Associated detection
 - Associated file
    
2) Outer Messages:  
 - Registration
 - Status
 - Detection
 - Alert
 - Error
 - Corresponding Acknowledgements 

| Type | Field name | Status | Protobuf Field Serial | Description |
|---|---|---|---:|---|
| NodeDefinition | node_definition | M List | 1 | Node type defines the broad category of node registering to allow the fusion node to handle the node appropriately |
| string | ICD_version | M | 2 | ICD version implemented by the node |
| string | name | D | 3 | Discretionary name for individual instance of node, e.g. car park camera |
| string | short_name | D | 4 | two-digit abbreviation of name, e.g. C1 for camera 1 |
| Capability | capabilities | M List | 5 | List of node capabilities, used to provide guidance to fusion nodes on what modalities/functionality a node contains |
| StatusDefinition | status_definition | M | 6 | — |
| ModeDefinition | mode_definition | M List | 7 | — |
| string | dependent_nodes | D List | 8 | Where the tasking of one node may also affect other nodes, e.g. a mobile platform or gimbal, the UUID of the dependent nodes is reported here |
| ConfigurationData | config_data | M List | 9 | Configuration data lists information to manage nodes and their configuration status |



## Useful Links

| Description | Link |
|-------------|------|
| Main ICD for Sapient | [ICD](https://www.bsigroup.com/siteassets/pdf/en/insights-and-media/insights/brochures/bsi-flex-335.pdf) |
| Overview of gRPC (where protobufs come from) | [gRPC](https://grpc.io/) |
| DSTL Repo for protobufs |[Protobuf](https://github.com/dstl/SAPIENT-Proto-Files) |
| DSTL Repo for test hardness | [Test Harness](https://github.com/dstl/BSI-Flex-335-v2-Test-Harness) |
| Overview of UUIDs | [UUID](https://www.rfc-editor.org/rfc/rfc9562.html) |